In [24]:
import pandas as pd

In [25]:
df=pd.read_csv("../Dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [26]:
def lowering(text):
    return text.lower()


df["lower_review"]=df["review"].apply(lowering)

In [27]:
df.head()

,review,sentiment,lower_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is..."


In [28]:
from bs4 import BeautifulSoup

def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()

df["clean_review"]=df["lower_review"].apply(remove_html_tags)

In [29]:
import string,time



def remove_punctuation(value):
    return value.translate(str.maketrans('','',string.punctuation))

df["clean_review"]=df["lower_review"].apply(remove_punctuation)

In [30]:
df.head()

,review,sentiment,lower_review,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,a wonderful little production br br the filmin...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...


In [31]:
from nltk.corpus import stopwords

stop=stopwords.words("english")

In [32]:
def stopword_removal(text):
    ll=[]
    for word in text.split():
        if word not in stop:
            ll.append(word)
    return " ".join(ll) 

In [33]:
df["clean_review"]=df["clean_review"].apply(stopword_removal)
df.head()

,review,sentiment,lower_review,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love time money visually stunni...


In [34]:
from nltk import WordNetLemmatizer

lemmatizer=WordNetLemmatizer()

def lemma(text):
    ll=[]
    for word in text.split():
        ll.append(lemmatizer.lemmatize(word))
    return " ".join(ll) 

In [ ]:
df["clean_review"]=df["clean_review"].apply(lemma)

In [36]:
df.head()

,review,sentiment,lower_review,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode y...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love time money visually stunni...


In [37]:
df=df.drop(columns=["review","lower_review"])

In [38]:
df.head()

,sentiment,clean_review
0,positive,one reviewer mentioned watching 1 oz episode y...
1,positive,wonderful little production br br filming tech...
2,positive,thought wonderful way spend time hot summer we...
3,negative,basically there family little boy jake think t...
4,positive,petter matteis love time money visually stunni...


In [39]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [ ]:
def convert_label(x):
    if(x=="positive"):
        return 1
    else:
        return 0
    
df["sentiment"]=df["sentiment"].apply(convert_label)

In [41]:
df.head()

,sentiment,clean_review
0,1,one reviewer mentioned watching 1 oz episode y...
1,1,wonderful little production br br filming tech...
2,1,thought wonderful way spend time hot summer we...
3,0,basically there family little boy jake think t...
4,1,petter matteis love time money visually stunni...


In [66]:
from sklearn.feature_extraction.text import CountVectorizer

vectorize=CountVectorizer(max_features=10000)

# We have to pass only text, no the dataframe. Thats why we specifically take Clean review column as X, not as a table

In [74]:
X=df["clean_review"]
Y=df["sentiment"]

In [75]:
X.head()

0    one reviewer mentioned watching 1 oz episode y...
1    wonderful little production br br filming tech...
2    thought wonderful way spend time hot summer we...
3    basically there family little boy jake think t...
4    petter matteis love time money visually stunni...
Name: clean_review, dtype: object

In [76]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=.25)

In [77]:
train_vec=vectorize.fit_transform(x_train)

In [78]:
train_vec[0].toarray()

array([[0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [79]:
from sklearn.naive_bayes import MultinomialNB

model=MultinomialNB()

In [80]:
model.fit(train_vec,y_train)

MultinomialNB()

In [81]:
trained_vec=vectorize.transform(x_test)

In [84]:
pred=model.predict(trained_vec)

In [86]:
from sklearn.metrics import classification_report

print(classification_report(pred,y_test))

              precision    recall  f1-score   support

           0       0.86      0.84      0.85      6402
           1       0.84      0.86      0.85      6098

    accuracy                           0.85     12500
   macro avg       0.85      0.85      0.85     12500
weighted avg       0.85      0.85      0.85     12500

